In [ ]:
from graphviz import Digraph
from IPython.display import Image, display

# Extract schema from SQLite via DBManager
with DBManager() as db:
	db.execute_query("""
		SELECT name
		FROM sqlite_master
		WHERE type='table' AND name NOT LIKE 'sqlite_%'
		ORDER BY name
	""")
	tables = [row[0] for row in db.cursor.fetchall()]

	schema = {}
	relations = []

	for table in tables:
		db.execute_query(f"PRAGMA table_info({table})")
		cols = db.cursor.fetchall()  # cid, name, type, notnull, dflt_value, pk
		schema[table] = cols

		db.execute_query(f"PRAGMA foreign_key_list({table})")
		fks = db.cursor.fetchall()  # id, seq, table, from, to, on_update, on_delete, match
		for fk in fks:
			relations.append((table, fk[3], fk[2], fk[4]))  # from_table, from_col, to_table, to_col

# Build diagram
dot = Digraph("database_schema", format="png")
dot.attr(rankdir="LR")

for table, cols in schema.items():
	col_lines = []
	for _, name, col_type, notnull, _, pk in cols:
		flags = []
		if pk:
			flags.append("PK")
		if notnull:
			flags.append("NOT NULL")
		suffix = f" ({', '.join(flags)})" if flags else ""
		col_lines.append(f"{name}: {col_type}{suffix}\\l")
	label = f"{{{table}|{''.join(col_lines)}}}"
	dot.node(table, label=label, shape="record")

for from_table, from_col, to_table, to_col in relations:
	dot.edge(from_table, to_table, label=f"{from_col} -> {to_col}")

# Export image
output_file = dot.render("db_schema", cleanup=True)
print(f"Schema image exported to: {output_file}")

# Preview in notebook
display(Image(filename=output_file))

In [ ]:
paire_id = 'JOMR'
import pandas as pd

from db_manager import DBManager
with DBManager() as db:

    subquery = f"""
        SELECT tp.point_id, tg.game_id, tp.team_a_score, tp.team_a_score_diff, tg.team_a, tg.team_b
        FROM table_game AS tg
        LEFT JOIN table_point AS tp ON tg.game_id = tp.game_id
        WHERE tg.team_a = '{paire_id}'"""

    db.execute_query(f"""SELECT ts.point_id, ts.player, ts.grade, ts.point_won, sub.game_id, sub.team_a_score, sub.team_a_score_diff, sub.team_b
                     FROM table_serve AS ts
                     LEFT JOIN ({subquery}) AS sub
                     ON ts.point_id = sub.point_id
                     WHERE ts.paire_id = '{paire_id}'
                     """)

    results = db.cursor.fetchall()

results_df = pd.DataFrame(results, columns=[desc[0] for desc in db.cursor.description])
results_df.head()

✅ Connection established: c:\Users\habib\Documents\GitHub\DataBeach\database\databeach_base.db
🔒 Connection closed.


,point_id,player,grade,point_won,game_id,team_a_score,team_a_score_diff,team_b
0,JOMR_jan26_MBV_01_p2,RANC Mathilde,out-of-system pass,0,JOMR_jan26_MBV_01,1,1,NoG_AliP
1,JOMR_jan26_MBV_01_p5,OFFREDI Jade,error,0,JOMR_jan26_MBV_01,2,0,NoG_AliP
2,JOMR_jan26_MBV_01_p7,RANC Mathilde,ace,1,JOMR_jan26_MBV_01,3,0,NoG_AliP
3,JOMR_jan26_MBV_01_p8,RANC Mathilde,ace,1,JOMR_jan26_MBV_01,4,1,NoG_AliP
4,JOMR_jan26_MBV_01_p9,RANC Mathilde,ace,1,JOMR_jan26_MBV_01,5,2,NoG_AliP


: 

In [5]:
from db_manager import DBManager
with DBManager() as db:
    db.execute_query(f"""SELECT point_id 
                     FROM table_serve 
                     WHERE grade = 'undetermined'
                     """)
    results = db.cursor.fetchall()

undetermined_points = [row[0] for row in results]
len(undetermined_points)

segmented_point_dir = r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\points_segmented'

import os
import shutil
os.makedirs(f'{segmented_point_dir}\\echange Thib-Jade', exist_ok=True)
for point_id in undetermined_points:
    # Build the full path of the source video file for the current point
    src = f'{segmented_point_dir}\\{point_id}.mp4'
    
    # Build the destination path inside the "echange Thib-Jade" subfolder
    dst = f'{segmented_point_dir}\\echange Thib-Jade\\{point_id}.mp4'
    
    # Copy the file only if it exists at the source location
    if os.path.exists(src):
        shutil.copy2(src, dst)



✅ Connection established: c:\Users\habib\Documents\GitHub\DataBeach\database\databeach_base.db
🔒 Connection closed.


In [30]:
csv_file = r'C:\Users\habib\Documents\GitHub\DataBeach\indexed_df_points\indexed_df_points_AleD-RonP_mar26_session_01.csv'
df = pd.read_csv(csv_file)
df['point_index'] = df['point_index'].apply(lambda x: str('999') if pd.isnull(x) else x)
df['point_index'] = df['point_index'].apply(lambda x: str(int(float(x))) if x != '999' else x)
df['point_index'] = df['point_index'].apply(lambda x: x.zfill(3) if x != '999' else x)
df.to_csv(r'C:\Users\habib\Documents\GitHub\DataBeach\indexed_df_points\indexed_df_points_AleD-RonP_mar26_session_01.csv', index=False)

In [3]:
from db_manager import DBManager
import pandas as pd

points_ids = [
    'JOMR_jan26_MBV_01_p013',
    'JOMR_jan26_MBV_01_p014',
    'JOMR_jan26_MBV_01_p015',
    'JOMR_jan26_MBV_01_p016',
    'JOMR_jan26_MBV_01_p017',
    ]
serve_or_pass = 'serve'

previous_grades_dict = dict()

with DBManager() as db:
    placeholders = ",".join(["?"] * len(points_ids))
    db.cursor.execute(
        f"""SELECT point_id, previous_grade
        FROM table_{serve_or_pass}
        WHERE point_id IN ({placeholders})""",
        points_ids,
    )
    results = db.cursor.fetchall()
    for point_id, previous_grade in results:
        previous_grades_dict[point_id] = previous_grade

previous_grades_dict
    

✅ Connection established: c:\Users\habib\Documents\GitHub\DataBeach\database\databeach_base.db
🔒 Connection closed.


{'JOMR_jan26_MBV_01_p013': 'ace',
 'JOMR_jan26_MBV_01_p014': 'good pass',
 'JOMR_jan26_MBV_01_p016': 'good pass'}

In [10]:
import pandas as pd
from db_manager import DBManager
with DBManager() as db:
    table_game_df = db.table_to_dataframe("table_game")
table_game_df

✅ Connection established: c:\Users\habib\Documents\GitHub\DataBeach\database\databeach_base.db
✅ Table 'table_game' exported: 30 rows, 12 columns
🔒 Connection closed.


,game_id,serie,stage,team_a,team_b,victory,set1_score,set2_score,set3_score,set1_score_adv,set2_score_adv,set3_score_adv
0,JOMR_jan26_MBV_01,MBV_S2-250_F_jan26,poule,JOMR,NoG_AliP,JOMR,,,,,,
1,JOMR_jan26_MBV_02,MBV_S2-250_F_jan26,poule,JOMR,AleN_EmP,JOMR,,,,,,
2,JOMR_jan26_MBV_03,MBV_S2-250_F_jan26,demi,JOMR,PauG_NeCH,JOMR,,,,,,
3,JOMR_jan26_MBV_04,MBV_S2-250_F_jan26,finale,JOMR,AleM_LauA,AleM_LauA,,,,,,
4,JOMR_nov24_Leuven_01,Leuven_Int_F_nov24,barrage,JOMR,(adv),(adv),,,,,,
5,JOMR_nov25_BSD_01,BSD_S2-250_F_nov25,demi,JOMR,MyH_HeR,JOMR,,,,,,
6,JOMR_nov25_BSD_02,BSD_S2-250_F_nov25,finale,JOMR,KaE_EmD,JOMR,,,,,,
7,JOMR_nov25_MBV_01,MBV_S2-500_F_nov25,quart,JOMR,ClaM_MaMeB,ClaM_MaMeB,,,,,,
8,JOMR_nov25_MBV_02,MBV_S2-500_F_nov25,barrage,JOMR,CloD_CirM,JOMR,,,,,,
9,JOMR_nov25_MBV_03,MBV_S2-500_F_nov25,poule,JOMR,VerF_FloA,JOMR,,,,,,


In [2]:
from video_edit_utils import cut_point_gpu

cut_point_gpu(
    video_path=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\JOMR_mar26_VSG_02_started_rotated.mp4',
    start_frame=36174,
    end_frame=50000,
    output_video=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\traitement adhoc\adhoc_video.mp4'
)

In [ ]:
from game_editor import GameEditor
game_editor = GameEditor(
    video_path=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\traitement adhoc\adhoc_video.mp4',
    output_dir=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\traitement adhoc\segmented_adhoc'
)
game_editor.game_to_segmented_points(
    team1_name='JOMR',
    team2_name='MarL_MarB',
)


In [8]:
import pandas as pd
adhoc_df = pd.read_csv(
    filepath_or_buffer=r'C:\Users\habib\Documents\GitHub\DataBeach\indexed_df_points\indexed_df_points_adhoc_video.csv',
    dtype={'point_index': str}
)
adhoc_df

,service_side,start_frame,end_frame,JOMR_score,MarL_MarB_score,JOMR_sets,MarL_MarB_sets,point_index
0,match start - service JOMR,89,363,0,0,0,0,001
1,service MarL_MarB,921,1164,0,1,0,0,002
2,service JOMR,1491,1690,1,1,0,0,003
3,service MarL_MarB,2059,2302,1,2,0,0,004
4,service MarL_MarB,2603,2931,1,3,0,0,005
5,*SWITCH*,3172,3203,1,3,0,0,999
6,Timeout,3597,3729,1,3,0,0,999
7,service MarL_MarB,4838,5133,1,4,0,0,006
8,service JOMR,5430,5700,2,4,0,0,007
9,service MarL_MarB,6127,6300,2,5,0,0,008


In [9]:
from video_edit_utils import extract_segments_from_df_gpu

extract_segments_from_df_gpu(
    video_path=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\traitement adhoc\adhoc_video.mp4',
    actions_df=adhoc_df,
    output_dir=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\traitement adhoc\segmented_adhoc'
)

game_id: adhoc_video
